In [ ]:
%sql
-- Script for generating comprehensive test data for Databricks environment

-- Create test data for the inventory movement table including diverse scenarios
CREATE OR REPLACE TEMP VIEW test_apl_qty_data AS
WITH test_data AS (
  -- Happy path test data
  SELECT '1' AS txn_id, 50.0 AS ref_txn_qty, 100.0 AS cumulative_txn_qty, 90.0 AS cumulative_ref_ord_sched_qty, 
    50.0 AS ref_ord_sched_qty, 40.0 AS prior_cumulative_txn_qty, 30.0 AS prior_cumulative_ref_ord_sched_qty, 40.0 AS apl_qty
  UNION ALL
  SELECT '2', -10.0, 80.0, 70.0, 40.0, 50.0, 45.0, -10.0
  UNION ALL
  SELECT '3', 20.0, 60.0, 100.0, 30.0, 30.0, 25.0, 30.0

  -- Edge cases
  UNION ALL
  SELECT '4', 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, NULL
  UNION ALL
  SELECT '5', 999.9, 9999.0, 9999.0, 999.9, 999.0, 999.0, 999.9
  
  -- Error cases
  UNION ALL
  SELECT '6', CAST('-invalid-' AS DECIMAL(3,1)), 100.0, 90.0, 50.0, 40.0, 30.0, NULL

  -- NULL handling scenarios
  UNION ALL
  SELECT '7', NULL, 100.0, 90.0, 50.0, 40.0, NULL, NULL

  -- Special characters and multi-byte characters
  UNION ALL
  SELECT '8', 55.5, 110.0, 100.0, 55.5, 45.0, 35.0, 45.5
)
SELECT * FROM test_data;

-- Insert the test data into the target table ensuring schema consistency
INSERT INTO purgo_playground.purgo_playground.f_inv_movmnt_apl_qty (
  txn_id, ref_txn_qty, cumulative_txn_qty, cumulative_ref_ord_sched_qty, 
  ref_ord_sched_qty, prior_cumulative_txn_qty, prior_cumulative_ref_ord_sched_qty, apl_qty
)
SELECT 
  txn_id, ref_txn_qty, cumulative_txn_qty, cumulative_ref_ord_sched_qty, 
  ref_ord_sched_qty, prior_cumulative_txn_qty, prior_cumulative_ref_ord_sched_qty, apl_qty
FROM test_apl_qty_data;

-- Data validation step using CTE to ensure correctness and handle any errors
WITH validated_data AS (
  SELECT * 
  FROM purgo_playground.purgo_playground.f_inv_movmnt_apl_qty
  WHERE ref_txn_qty IS NOT NULL 
    AND cumulative_txn_qty >= 0
    AND cumulative_ref_ord_sched_qty >= 0
)
SELECT txn_id, apl_qty
FROM validated_data;

